# Necessary Imports

In [1]:
import numpy as np
import pandas as pd

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import regex as re
from bs4 import BeautifulSoup
import time

# Getting Product Links from a Category Page

In [ ]:
# To open a new window: webdriver.Chrome(here pass the driver as a service)
# we need to create a object of a service class so that we can pass it to open a chrome window 
s=Service("C:/Users/Tanvir Ahmed/Downloads/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=s)
#after opening the window, searching for the link
driver.get('https://www.zazzle.com/s/bow+graduation+invitations?st=orderitemcount_month')
time.sleep(2) #wait 2 seconds
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)") #scroll to the bottom to load all product
time.sleep(3)
page=driver.page_source #getting the source code of that loaded page
soup = BeautifulSoup(page,'lxml') #create a soup object to parse it, to get all products link
driver.quit() #close the window

In [ ]:
links=soup.find_all('a',{'class':'Link Link--marketplaceTheme SearchResultsGridCell2_link'}) #getting all product links available in the page by selecting specific class
link=[]
for i in range(0,len(links),2): #there are double link for a single product scrapped, that's why I just took every alternatinve links
    link.append(links[i]['href']) #getting the only links
link

['https://www.zazzle.com/pink_bow_photo_2025_graduation_invitation-256055896693503895',
 'https://www.zazzle.com/pink_bow_photo_graduation_party_invitation-256093504405710144',
 'https://www.zazzle.com/wildflower_pink_bow_graduation_invitation-256216092353790857',
 'https://www.zazzle.com/trendy_modern_photo_graduation_invitation-256920141544615649',
 'https://www.zazzle.com/elegant_black_bow_photo_grad_party_invitation-256525922731407841',
 'https://www.zazzle.com/arch_pink_bow_photo_graduation_party_invitation-256286770993629234',
 'https://www.zazzle.com/modern_trendy_pink_bow_graduation_cap_graduation_invitation-256039033918865893',
 'https://www.zazzle.com/wildflower_bow_graduation_party_photo_collage_invitation-256845528064435196',
 'https://www.zazzle.com/modern_elegant_pink_bow_graduation_cap_graduation_invitation-256204289868702558',
 'https://www.zazzle.com/black_bow_photo_graduation_invitation-256161264020359462',
 'https://www.zazzle.com/pink_bow_graduation_invitation-25695

# Scraping Data From Each Prodcut

In [ ]:
s=Service("C:/Users/Tanvir Ahmed/Downloads/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=s)

title=[]
view=[]
created_date=[]
tag=[]

wait = WebDriverWait(driver, 10) #set a waiter

for i in link:
    driver.get(i) #visiting each link

    # wait until page loads the title
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "ProductSpaceDetailsPod_title")))
    except:
        print(f"Page not loaded properly: {i}")
        continue
    #scroll to the end of the page
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    
    #wait until page loads the OtherInfo block
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "OtherInfo")))
    except:
        print(f"Page not loaded properly: {i}")
        continue    

    #after loead all information get the source html of the page
    html=driver.page_source

    #now I can convert it to a soup object and then parse it to get all information
    soup=BeautifulSoup(html,'lxml')

    #get the title
    try:
        t = soup.find('h1', {'class': 'ProductSpaceDetailsPod_title'}).text.strip()
    except:
        t = None
    title.append(t)

    #get the total views
    try:
        v = soup.find('div', {'class': 'BehavioralCallout_copy'}).find_all('div')[1].text.strip()
    except:
        v = None
    view.append(v)

    #get the created on date
    try:
        d = soup.find('div', {'class': 'OtherInfo'}).find_all('div')[1].find('span').text.strip()
    except:
        d = None
    created_date.append(d)

    #get all tags and merge them into a string
    try:
        all_tags = soup.find('div', {'class': 'Tags-tagsList'}).find_all('span')
        s = [tg.text.strip() for tg in all_tags]
        s = ", ".join(s)
    except:
        s = None
    tag.append(s)

#close browser
driver.quit()

#store those data into a dataframe
df=pd.DataFrame({
    "Link":link,
    'Title':title,
    'View':view,
    'Created Date':created_date,
    'Tag':tag
})


In [ ]:
#making backups
df_original=df.copy()

# Data Cleaning

In [108]:
#converting the date to datetime format and sort the dataframe on date ( resent product comes first)
df["Created Date"] = pd.to_datetime(
    df["Created Date"],
    format="%m/%d/%Y, %I:%M %p",
    errors="coerce"
)
df = df.sort_values(by="Created Date", ascending=False)
df

,Link,Title,View,Created Date,Tag
53,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,28 people viewed this design,2026-03-20 13:18:00,"graduation daughter, coquette invite, arch pho..."
7,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower Bow Graduation party Photo Collage ...,612 people viewed this design,2026-03-14 02:32:00,"graduation party, grad, wildflower, floral, gi..."
57,https://www.zazzle.com/wildflower_graduation_s...,Wildflower Graduation Sign,40 people viewed this design,2026-03-03 05:04:00,"wildflower graduation invitation, pink bow, gr..."
55,https://www.zazzle.com/she_did_it_graduation_p...,She Did It! Graduation Party Invitation,37 people viewed this design,2026-03-02 15:13:00,"chic party invitations for her, elegant party ..."
9,https://www.zazzle.com/black_bow_photo_graduat...,Black Bow Photo Graduation Invitation,364 people viewed this design,2026-02-24 17:48:00,"graduation daughter, coquette invite, arch pho..."
21,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,194 people viewed this design,2026-02-24 17:22:00,"graduation daughter, coquette invite, elegant ..."
17,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,189 people viewed this design,2026-02-24 16:12:00,"graduation daughter, coquette invite, arch pho..."
39,https://www.zazzle.com/wildflower_graduation_p...,Wildflower Graduation Party Return Address Label,45 people viewed this design,2026-02-24 00:01:00,"graduation invitation, wildflower, floral, gra..."
46,https://www.zazzle.com/pink_bow_in_her_grad_er...,Pink Bow In Her Grad Era Elegant Photo Graduat...,176 people viewed this design,2026-02-19 02:40:00,"pink bow graduation party invitation, in her g..."
56,https://www.zazzle.com/black_bow_photo_graduat...,Black Bow Photo Graduation Invitation,221 people viewed this design,2026-02-05 22:57:00,"black bow graduation, graduation daughter, coq..."


In [ ]:
# make the view column better, string to number
def string_to_int(text):
    match = re.search(r"\d+(\.\d+)?", str(text)) #search for the number
    
    if not match:
        return None
    
    number = float(match.group()) #extract the number
    
    if "K" in str(text): #if there any "K" in the string mulply the number with 1000
        return int(number * 1000)
    else:
        return int(number)

In [110]:
df['View']=df['View'].apply(string_to_int)

# End

## Making a Frequency Table for the Tags

In [ ]:
tag_df = (
    df["Tag"]
    .str.split(", ") #split the tags from a single string to array of tags
    .explode() #flatten the tags, from array to each tags at a line
    .value_counts() #calculate the frequency count
)

In [140]:
tag_df.head(60)

Tag
modern                                     17
coquette                                   14
trendy                                     14
class of 2026                              14
grad party                                 14
arch photo                                 13
graduate                                   12
graduation                                 10
calligraphy script typography classy        9
elegant                                     9
high school college                         8
pink bow                                    8
bow                                         8
high school college graduate university     8
graduation invitations announcements        7
casual minimalist simple grad party         7
coquette invite                             7
graduation daughter                         7
elegant bow                                 7
graduation party                            7
high school or college                      6
floral                        

In [ ]:
#now using that frequency table assigning those frequency value to each tags on the dataframe also assign the total frequency count at the end of the each product's tag
tag_dict = tag_df.to_dict()
def tag_with_freq_and_total(tag_string):
    tags = [t.strip() for t in tag_string.split(",")] #converting string to list of tags
    
    counts = [tag_dict.get(tag, 0) for tag in tags] #getting the freq count for each tags and store it in a list
    
    # tag + freq
    tag_freq = [f"{tag} {count}" for tag, count in zip(tags, counts)] # assining the frequency count along with each tag
    
    total = sum(counts)
    
    return ", ".join(tag_freq) + f" | Total: {total}" #returting the final string with total counts

In [143]:
df["Tag_with_freq"] = df["Tag"].apply(tag_with_freq_and_total)
df

,Link,Title,View,Created Date,Tag,Tag_with_freq
53,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,28,2026-03-20 13:18:00,"graduation daughter, coquette invite, arch pho...","graduation daughter 7, coquette invite 7, arch..."
7,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower Bow Graduation party Photo Collage ...,612,2026-03-14 02:32:00,"graduation party, grad, wildflower, floral, gi...","graduation party 7, grad 2, wildflower 4, flor..."
57,https://www.zazzle.com/wildflower_graduation_s...,Wildflower Graduation Sign,40,2026-03-03 05:04:00,"wildflower graduation invitation, pink bow, gr...","wildflower graduation invitation 1, pink bow 8..."
55,https://www.zazzle.com/she_did_it_graduation_p...,She Did It! Graduation Party Invitation,37,2026-03-02 15:13:00,"chic party invitations for her, elegant party ...","chic party invitations for her 1, elegant part..."
9,https://www.zazzle.com/black_bow_photo_graduat...,Black Bow Photo Graduation Invitation,364,2026-02-24 17:48:00,"graduation daughter, coquette invite, arch pho...","graduation daughter 7, coquette invite 7, arch..."
21,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,194,2026-02-24 17:22:00,"graduation daughter, coquette invite, elegant ...","graduation daughter 7, coquette invite 7, eleg..."
17,https://www.zazzle.com/wildflower_bow_graduati...,Wildflower & Bow Graduation Invitation,189,2026-02-24 16:12:00,"graduation daughter, coquette invite, arch pho...","graduation daughter 7, coquette invite 7, arch..."
39,https://www.zazzle.com/wildflower_graduation_p...,Wildflower Graduation Party Return Address Label,45,2026-02-24 00:01:00,"graduation invitation, wildflower, floral, gra...","graduation invitation 1, wildflower 4, floral ..."
46,https://www.zazzle.com/pink_bow_in_her_grad_er...,Pink Bow In Her Grad Era Elegant Photo Graduat...,176,2026-02-19 02:40:00,"pink bow graduation party invitation, in her g...","pink bow graduation party invitation 1, in her..."
56,https://www.zazzle.com/black_bow_photo_graduat...,Black Bow Photo Graduation Invitation,221,2026-02-05 22:57:00,"black bow graduation, graduation daughter, coq...","black bow graduation 1, graduation daughter 7,..."


In [ ]:
df.to_csv("Bow Graduation Invitatoin.csv") #save it to a csv file